# LAB 03
Janetzy Giselle Tino Flores Aviña

In [1]:
from spark_utils import SparkUtils
spark_url = "spark://spark-master:7077"
app_name = "Lab 03"
su = SparkUtils(spark_url, app_name)
su

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/24 02:41:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/02/24 02:41:17 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


<SparkContext master=local[*] appName=Spark SQL example>

Create the Quick Commerce Schema

In [2]:


from pyspark.sql.types import StructType, StringType, IntegerType, IntegerType, LongType, ShortType, DoubleType, FloatType, BooleanType, DateType, FloatType, BooleanType, DateType, TimestampType, BinaryType, StructField, ArrayType

 

In [3]:
"""
Order_ID,Company,City,Customer_Age,Order_Value,Delivery_Time_Min,Distance_Km,Items_Count,
Product_Category,Payment_Method,Customer_Rating,Discount_Applied,Delivery_Partner_Rating
"""
columns_types = [("Order_ID", "long"),
                ("Company", "string"),
                ("City", "string"), 
                ("Customer_Age", "integer"),
                ("Order_Value", "float"),
                ("Delivery_Time_Min", "float"),
                ("Distance_Km", "float"),
                ("Items_Count", "float"),
                ("Product_Category", "string"),
                ("Payment_Method", "string"),
                ("Customer_Rating", "float"),
                ("Discount_Applied", "float"),
                ("Delivery_Partner_Rating", "float")
]

qcommerce_schema = SparkUtils.generate_schema(columns_types)

qcommerce_df = su._spark \
                .read \
                .option("header", "true") \
                .schema(qcommerce_schema) \
                .csv("/opt/spark/work-dir/data/qcommerce/")
qcommerce_df.printSchema()
qcommerce_df.show(5)


root
 |-- Order_ID: long (nullable = true)
 |-- Company: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Customer_Age: integer (nullable = true)
 |-- Order_Value: float (nullable = true)
 |-- Delivery_Time_Min: float (nullable = true)
 |-- Distance_Km: float (nullable = true)
 |-- Items_Count: float (nullable = true)
 |-- Product_Category: string (nullable = true)
 |-- Payment_Method: string (nullable = true)
 |-- Customer_Rating: float (nullable = true)
 |-- Discount_Applied: float (nullable = true)
 |-- Delivery_Partner_Rating: float (nullable = true)

+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+
|Order_ID|         Company|    City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|  Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|
+--------+----------------+----

In [4]:
from pyspark.sql.functions import when, count, isnull
qcommerce_df.columns

['Order_ID',
 'Company',
 'City',
 'Customer_Age',
 'Order_Value',
 'Delivery_Time_Min',
 'Distance_Km',
 'Items_Count',
 'Product_Category',
 'Payment_Method',
 'Customer_Rating',
 'Discount_Applied',
 'Delivery_Partner_Rating']

In [5]:
qcommerce_null_count_df = qcommerce_df.select([count(when(isnull(c), c)).alias(c) for c in qcommerce_df.columns])
qcommerce_null_count_df.show()

[Stage 1:=======>                                                 (3 + 19) / 22]

+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+
|Order_ID|Company| City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|
+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+
|       0|      0|52000|           0|          0|                0|          0|      35000|               0|             0|          47000|               0|                 104137|
+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+



In [6]:
qcommerce_null_count_df_2 = SparkUtils.count_nulls(qcommerce_df)
qcommerce_null_count_df_2.show()

[Stage 4:==>                                                      (1 + 21) / 22]

+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+
|Order_ID|Company| City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|
+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+
|       0|      0|52000|           0|          0|                0|          0|      35000|               0|             0|          47000|               0|                 104137|
+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+



In [7]:
qcommerce_wo_nulls = qcommerce_df.dropna()
print(f"size of df after removing nulls: {qcommerce_wo_nulls.count()}")

[Stage 7:=====>                                                   (2 + 20) / 22]

size of df after removing nulls: 780992


In [8]:
qcommerce_wo_nulls_fillna = qcommerce_df.fillna({
    'City': 'Unknown',
    'Items_count': 0.0,
    'Customer_Rating': 0.0,
    'Delivery_Partner_Rating': 0.0
})

print(f"size  of df after removing nulls with fillna: {qcommerce_wo_nulls_fillna.count()}")

[Stage 10:==>                                                     (1 + 21) / 22]

size  of df after removing nulls with fillna: 1000000


In [9]:
from pyspark.sql. functions import lit
qcommerce_wo_nulls_fillna = qcommerce_wo_nulls_fillna.withColumn("Code_Country", lit("IN"))
qcommerce_wo_nulls_fillna.show(5)

+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+------------+
|Order_ID|         Company|    City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|  Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|Code_Country|
+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+------------+
| 1000001|Swiggy Instamart|   Noida|          46|   702.3375|           19.182|      11.97|       12.0|           Dairy|          Wallet|            2.1|             1.0|                    3.2|          IN|
| 1000002|Flipkart Minutes|Amritsar|          56|     1007.3|           19.644|      12.74|       10.0|          Snacks|Cash on Delivery|            2.3|             0.

In [10]:
from pyspark.sql. functions import col
qcommerce_tax_df = qcommerce_wo_nulls_fillna.withColumn("Paid_Tax", col("Order_Value") * 0.16)
qcommerce_tax_df.show(5)

+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+------------+----------------+
|Order_ID|         Company|    City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|  Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|Code_Country|        Paid_Tax|
+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+------------+----------------+
| 1000001|Swiggy Instamart|   Noida|          46|   702.3375|           19.182|      11.97|       12.0|           Dairy|          Wallet|            2.1|             1.0|                    3.2|          IN| 112.37400390625|
| 1000002|Flipkart Minutes|Amritsar|          56|     1007.3|           19.644|      12.74|       10

### Lab 03

# TASK 1
Create Delivery_SLA with withColumn + when:
- Delivery_Time_Min ≤ 15 → "FAST", 15--20 → "ON_TIME", > 20 → "LATE"
- filter only Delivery_SLA = "LATE" and orderBy Delivery_Time_Min (desc).
- Display: Order_ID, Company, City, Delivery_Time_Min, Delivery_SLA

In [25]:
from pyspark.sql import functions

qcommerce_df_taks1= qcommerce_tax_df.withColumn("Delivery_SLA", when(col("Delivery_Time_Min") <= 15, "FAST") \
                                                .when((col("Delivery_Time_Min") > 15) & (col("Delivery_Time_Min") <= 20),
                                                "ON-TIME") \
                                                .when(col("Delivery_Time_Min") > 20, "LATE")) \
                                                .filter(col("Delivery_SLA") == "LATE") \
                                                .orderBy(col("Delivery_Time_Min"), ascending=False)
qcommerce_df_taks1.select("Order_ID", "Company", "City", "Delivery_Time_Min", "Delivery_SLA", "Customer_Age").show()

+--------+--------+--------+-----------------+------------+------------+
|Order_ID| Company|    City|Delivery_Time_Min|Delivery_SLA|Customer_Age|
+--------+--------+--------+-----------------+------------+------------+
| 1529779|Jio Mart|Haridwar|             40.0|        LATE|          41|
| 1807126|Jio Mart|Haridwar|             40.0|        LATE|          54|
| 1610720|Jio Mart|Haridwar|           39.994|        LATE|          32|
| 1729503|Jio Mart|Haridwar|           39.994|        LATE|          42|
| 1165764|Jio Mart|Haridwar|           39.994|        LATE|          20|
| 1951122|Jio Mart|Haridwar|           39.988|        LATE|          20|
| 1975896|Jio Mart|Haridwar|           39.988|        LATE|          54|
| 1826295|Jio Mart|Haridwar|           39.982|        LATE|          43|
| 1594835|Jio Mart|Haridwar|           39.982|        LATE|          40|
| 1059671|Jio Mart|Haridwar|           39.982|        LATE|          34|
| 1162804|Jio Mart|Haridwar|           39.982|     

# TASK 2
Create Effective_Order_Value = Order_Value * (1 - Discount_Applied).
- Create Price_Bucket using when:
    < 200 → “LOW”    
    200–500 → “MEDIUM”   
    500 → “HIGH”
- Group by Price_Bucket and compute:
    count(*)
    avg(Effective_Order_Value)
- Order by average effective value (descending).

In [16]:
from pyspark.sql.functions import avg, count
qcommerce_task2_df = qcommerce_df_taks1 \
        .withColumn("Effective_Order_Value", col("Order_Value") * (1 - col("Discount_Applied"))) \
        .withColumn("Price_Bucket", when(col("Effective_Order_Value") < 200, "LOW")\
                                    .when((col("Effective_Order_Value") >= 200) & (col("Effective_Order_Value") <= 500), "MEDIUM")
                                    .otherwise("HIGH")
    )

result_task2 = qcommerce_task2_df \
    .groupBy("Price_Bucket").agg(
        count("*").alias("Count_Price_Bucket"),
        avg("Effective_Order_Value").alias("avg_effective_order_value")
    ) \
    .orderBy(col("avg_effective_order_value").desc())

result_task2.show()

[Stage 20:=======>                                                (3 + 19) / 22]

+------------+------------------+-------------------------+
|Price_Bucket|Count_Price_Bucket|avg_effective_order_value|
+------------+------------------+-------------------------+
|        HIGH|             56013|         707.219837039914|
|      MEDIUM|             59824|       353.49134017350553|
|         LOW|            143811|        25.36497016341368|
+------------+------------------+-------------------------+



# TASK 3
- Create Age_Group using withColumn + when:
    < 25 → “YOUNG”
    25–44 → “ADULT”
    ≥ 45 → “SENIOR”
- Filter invalid ages (nulls, < 0, or > 100).
    Group by Age_Group and Product_Category and compute:
    orders = count(*)
    avg_order_value = avg(Order_Value)
- Order by Age_Group (ascending) and orders (descending) to find top categories per segment.

In [26]:
qcommerce_task3_df = qcommerce_df_taks1.withColumn(
        "Age_Group",
        when(col("Customer_Age") < 25, "YOUNG")
        .when((col("Customer_Age") >= 25) & (col("Customer_Age") <= 44), "ADULT")
        .otherwise("SENIOR")
    )

result_task3 = qcommerce_task3_df \
    .groupBy("Age_Group", "Product_Category") \
    .agg(
        count("*").alias("orders"),
        avg("Order_Value").alias("avg_order_value")
    ) \
    .orderBy(col("Age_Group").asc(), col("orders").desc())

result_task3.show()

[Stage 68:=====>                                                  (2 + 20) / 22]

+---------+-------------------+------+------------------+
|Age_Group|   Product_Category|orders|   avg_order_value|
+---------+-------------------+------+------------------+
|    ADULT|      Personal Care| 17935|495.73422188930334|
|    ADULT|              Dairy| 17780| 502.4952578519943|
|    ADULT|          Household| 17767|497.23453261697006|
|    ADULT|             Snacks| 17700|499.79997128481244|
|    ADULT|Fruits & Vegetables| 17671|494.20056196792063|
|    ADULT|          Beverages| 17650| 496.7228836926252|
|    ADULT|          Groceries| 17545|491.13630949723205|
|   SENIOR|          Groceries| 13440|500.87560482649576|
|   SENIOR|      Personal Care| 13253| 501.4177858687559|
|   SENIOR|             Snacks| 13250|502.81940932795686|
|   SENIOR|              Dairy| 13176|498.58672649583895|
|   SENIOR|          Household| 13163|  489.620170382284|
|   SENIOR|          Beverages| 13118|492.35374550340305|
|   SENIOR|Fruits & Vegetables| 12955|498.62873171412866|
|    YOUNG|   

In [27]:
su._spark.stop()